In [11]:
#Step 2 Preprocessing and Feature Engineering
#Purpose Build features used by PD, LGD, and EAD models. Typical features: credit score, DPD (days past due), LTV, collateral coverage, repayment consistency, trade exposure, macro indicators.
# features.py
import pandas as pd
import numpy as np
import os

os.getcwd()
os.chdir(r'C:\GYANENDRA\INFORMATION_TECHNILOGY_PROJECTS\CREDIT_RISK_ENGINE')


BASE = "CREDIT_RISK_CLIENT_DATA"

clients = pd.read_csv(f"{BASE}/counterparty.csv", sep=";")
loans = pd.read_csv(f"{BASE}/credit_data.csv", sep=";")
coll = pd.read_csv(f"{BASE}/collateral.csv", sep=";")
rep = pd.read_csv(f"{BASE}/repayment.csv", sep=";")
trades = pd.read_csv(f"{BASE}/trades.csv", sep=";")
trades.head()


,TradeID,ClientID,TradeVolume_Cr,TradeFrequency,LastTradeDate,AssetClass,MarketExposure_Cr,RiskAppetiteScore
0,T000001,C00001,15.48,28,2026-08-15,Equity,81.54,78
1,T000002,C00001,35.06,16,2025-12-19,Bonds,85.16,41
2,T000003,C00001,11.86,8,2026-01-24,Bonds,80.81,53
3,T000004,C00001,9.86,47,2025-10-03,Derivatives,95.84,63
4,T000005,C00002,16.59,37,2025-12-25,Bonds,38.12,58


In [22]:
# Aggregate collateral per loan
coll_agg = coll.groupby("LoanID").agg({
    "CollateralValue_Cr":"sum",
    "HaircutPercent":"mean"
}).rename(columns={"CollateralValue_Cr":"TotalCollateral_Cr","HaircutPercent":"AvgHaircut"})

# Latest repayment status per loan
rep_latest = rep.sort_values("LastPaymentDate").groupby("LoanID").tail(1).set_index("LoanID")

# Trade exposure per client
trade_agg = trades.groupby("ClientID").agg({"TradeVolume_Cr":"sum","TradeFrequency":"sum"}).fillna(0)

# Merge features
df = loans.merge(clients, on="ClientID", how="left")
df = df.merge(coll_agg, on="LoanID", how="left").merge(rep_latest[["PaymentStatus","DaysPastDue","OutstandingBalance_Cr"]], on="LoanID", how="left")
df = df.merge(trade_agg, on="ClientID", how="left").fillna(0)

# Derived features
df["LTV"] = df["OutstandingBalance_Cr"] / (df["TotalCollateral_Cr"].replace(0,np.nan))
df["LTV"] = df["LTV"].fillna(10)  # unsecured or missing collateral
df["DPD"] = df["DaysPastDue"]
df["CreditScore_norm"] = (df["CreditScore"] - df["CreditScore"].mean()) / df["CreditScore"].std()
df["RepaymentRisk"] = df["PaymentStatus"].map({"Paid":0,"Overdue":1,"Default":2}).fillna(0)
df.to_csv(f"{BASE}/features.csv", sep=";", index=False)
print("Features saved")


Features saved


In [23]:
df

,LoanID,ClientID,LoanAmount_Cr,InterestRate,Tenure_Months,StartDate,EndDate,LoanType,RiskCategory,PurposeOfLoan,...,AvgHaircut,PaymentStatus,DaysPastDue,OutstandingBalance_Cr,TradeVolume_Cr,TradeFrequency,LTV,DPD,CreditScore_norm,RepaymentRisk
0,L000001,C00001,62.15,10.97,12,2022-01-23,2031-07-30,Secured,High,Mortgage,...,20.000000,Default,6,87.61,72.26,99,0.507884,6,-1.238435,2
1,L000002,C00001,2.00,10.00,12,2023-09-11,2031-06-19,Secured,Medium,Business,...,20.000000,Default,49,23.29,72.26,99,0.123699,49,-1.238435,2
2,L000003,C00001,37.82,7.34,36,2021-11-17,2030-11-18,Unsecured,High,Business,...,20.000000,Overdue,55,35.22,72.26,99,1.822038,55,-1.238435,1
3,L000004,C00002,95.03,7.35,48,2026-08-03,2027-02-06,Unsecured,Medium,Personal,...,25.000000,Default,61,25.64,16.59,37,0.145047,61,-1.629890,2
4,L000005,C00003,52.05,9.30,48,2025-11-05,2029-11-02,Secured,Low,Mortgage,...,10.000000,Paid,0,23.65,119.84,76,0.964126,0,-1.721997,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21149,L021150,C09999,77.48,11.31,36,2024-08-26,2029-02-25,Unsecured,Medium,Mortgage,...,10.000000,Default,8,92.01,78.59,89,0.316012,8,-0.490064,2
21150,L021151,C09999,75.45,11.73,12,2022-04-15,2030-07-29,Secured,Low,Personal,...,23.333333,Default,38,75.97,78.59,89,0.298507,38,-0.490064,2
21151,L021152,C10000,20.26,7.51,24,2022-06-26,2029-01-23,Unsecured,High,Personal,...,10.000000,Paid,0,57.62,42.27,57,0.280280,0,-0.467037,0
21152,L021153,C10000,54.08,8.21,12,2023-06-27,2029-04-28,Unsecured,Medium,Business,...,30.000000,Default,78,84.63,42.27,57,0.569018,78,-0.467037,2


In [25]:
#Step 3 Risk Scoring Models PD LGD EAD
#Purpose Train simple models for Probability of Default (PD) and Loss Given Default (LGD). Use logistic regression for PD and a regression model for LGD. Save models with joblib.

# train_models.py
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
import joblib

df = pd.read_csv(f"{BASE}/features.csv", sep=";")

# Create synthetic target PD: default if RepaymentRisk >=1 and DPD>30
df["default_flag"] = ((df["RepaymentRisk"]>=1) & (df["DPD"]>30)).astype(int)
# Synthetic LGD target: 1 - (collateral / exposure) clipped
df["lgd_target"] = np.clip(1 - (df["TotalCollateral_Cr"] / (df["OutstandingBalance_Cr"].replace(0,1))), 0, 1)

features = ["LoanAmount_Cr","InterestRate","Tenure_Months","LTV","DPD","CreditScore_norm","TradeVolume_Cr"]
X = df[features].fillna(0)
y_pd = df["default_flag"]
y_lgd = df["lgd_target"]

# PD model
X_train, X_test, y_train, y_test = train_test_split(X, y_pd, test_size=0.2, random_state=42)
pd_model = LogisticRegression(max_iter=200)
pd_model.fit(X_train, y_train)
print("PD AUC", roc_auc_score(y_test, pd_model.predict_proba(X_test)[:,1]))

# LGD model
X_train, X_test, y_train, y_test = train_test_split(X, y_lgd, test_size=0.2, random_state=42)
lgd_model = Ridge()
lgd_model.fit(X_train, y_train)

#print("LGD RMSE", mean_squared_error(y_test, lgd_model.predict(X_test), squared=False))

from sklearn.metrics import mean_squared_error
preds = lgd_model.predict(X_test)
rmse = mean_squared_error(y_test, preds) ** 0.5
print("LGD RMSE", rmse)


joblib.dump(pd_model, f"{BASE}/pd_model.joblib")
joblib.dump(lgd_model, f"{BASE}/lgd_model.joblib")
print("Models saved")



PD AUC 1.0
LGD RMSE 0.11641468638226718
Models saved


In [26]:
#Step 4 Monitoring Metrics and Alerts
#Purpose Compute portfolio level metrics and generate alerts when thresholds breach. Typical metrics: portfolio PD, expected loss (EL = PD * LGD * EAD), vintage delinquency rates, concentration metrics.
#python
# monitor.py (core monitoring functions)

import pandas as pd
import joblib
import numpy as np

BASE="CREDIT_RISK_CLIENT_DATA"

df = pd.read_csv(f"{BASE}/features.csv", sep=";")
pd_model = joblib.load(f"{BASE}/pd_model.joblib")
lgd_model = joblib.load(f"{BASE}/lgd_model.joblib")

features = ["LoanAmount_Cr","InterestRate","Tenure_Months","LTV","DPD","CreditScore_norm","TradeVolume_Cr"]
X = df[features].fillna(0)

df["PD"] = pd_model.predict_proba(X)[:,1]
df["LGD"] = np.clip(lgd_model.predict(X), 0, 1)
df["EAD"] = df["OutstandingBalance_Cr"]
df["ExpectedLoss"] = df["PD"] * df["LGD"] * df["EAD"]

# Portfolio metrics
portfolio_pd = (df["PD"] * df["EAD"]).sum() / df["EAD"].sum()
portfolio_el = df["ExpectedLoss"].sum()
avg_lgd = df["LGD"].mean()

print(f"Portfolio PD weighted {portfolio_pd:.4f}")
print(f"Portfolio Expected Loss {portfolio_el:.2f} Cr")
print(f"Average LGD {avg_lgd:.2%}")

# Alerts
alerts=[]
if portfolio_pd > 0.05:
    alerts.append(("High Portfolio PD", f"Weighted PD {portfolio_pd:.2%} exceeds 5% threshold"))
if portfolio_el > 100:  # example threshold
    alerts.append(("High Expected Loss", f"EL {portfolio_el:.2f} Cr exceeds 100 Cr"))

# Top concentrations
top_clients = df.groupby("ClientID")["EAD"].sum().sort_values(ascending=False).head(10)
print("Top client exposures")
print(top_clients)

for a in alerts:
    print("ALERT:", a[0], "-", a[1])

Portfolio PD weighted 0.4448
Portfolio Expected Loss 24302.93 Cr
Average LGD 4.20%
Top client exposures
ClientID
C04201    647.79
C07324    646.00
C00626    528.94
C01612    528.86
C00381    515.71
C00088    514.07
C00519    471.02
C06977    454.56
C06092    444.36
C00813    442.37
Name: EAD, dtype: float64
ALERT: High Portfolio PD - Weighted PD 44.48% exceeds 5% threshold
ALERT: High Expected Loss - EL 24302.93 Cr exceeds 100 Cr


In [ ]:
#Alerting options
#Print to console (above)
#Send email via SMTP
#Push to Slack or webhook
#Create tickets in monitoring system
#Step 5 Dashboard and Reporting
#Purpose Provide a quick Streamlit dashboard to inspect portfolio metrics, top exposures, and time series of EL.
#python
# streamlit_dashboard.py

import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt

st.title("Credit Risk Monitoring Dashboard")
BASE="risk_data"
df = pd.read_csv(f"{BASE}/features.csv", sep=";")
pd_model = joblib.load(f"{BASE}/pd_model.joblib")
lgd_model = joblib.load(f"{BASE}/lgd_model.joblib")

features = ["LoanAmount_Cr","InterestRate","Tenure_Months","LTV","DPD","CreditScore_norm","TradeVolume_Cr"]
X = df[features].fillna(0)
df["PD"] = pd_model.predict_proba(X)[:,1]
df["LGD"] = lgd_model.predict(X).clip(0,1)
df["EAD"] = df["OutstandingBalance_Cr"]
df["EL"] = df["PD"] * df["LGD"] * df["EAD"]

st.metric("Portfolio Expected Loss", f"{df['EL'].sum():.2f} Cr")
st.metric("Weighted PD", f"{(df['PD']*df['EAD']).sum()/df['EAD'].sum():.2%}")

st.subheader("Top 10 Client Exposures")
top = df.groupby("ClientID")["EAD"].sum().sort_values(ascending=False).head(10).reset_index()
st.table(top)

st.subheader("EL Distribution")
fig, ax = plt.subplots()
ax.hist(df["EL"].clip(0,10), bins=50)
st.pyplot(fig)

In [ ]:
streamlit run streamlit_dashboard.py